In [1]:
import backtrader as bt
import datetime
import pandas as pd


In [2]:
dataframe = pd.read_csv('./csv_files/000300.XSHG.csv', parse_dates=True, index_col=0)

# pandasdata feeder
feed = bt.feeds.PandasData(dataname=dataframe, openinterest=None)

In [3]:
# Create a Stratey
class MACDStrategy(bt.Strategy):
    params = (
        ('fastperiod', 10),
        ('slowperiod', 22),
        ('signalperiod', 8),
    )
    
    def log(self, txt):
        ''' Logging function for this strategy'''
        dt = self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))
        
    def __init__(self):
        self.macd = bt.indicators.MACDHisto(
            self.data0,
            period_me1=self.p.fastperiod,
            period_me2=self.p.slowperiod,
            period_signal=self.p.signalperiod,
        )
    
        self.cross = bt.indicators.CrossOver(self.macd.macd, self.macd.signal, plot=False)
        self.above = bt.And(self.macd.macd>0.0, self.macd.signal>0.0)
        self.buy_signal = bt.And(self.above, self.cross==1)
        self.sell_signal = self.cross==-1
        
        # To keep track of pending orders
        self.order = None
        
    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            # Buy/Sell order submitted/accepted to/by broker - Nothing to do
            return

        # Check if an order has been completed
        # Attention: broker could reject order if not enough cash
#         if order.status == order.Completed:
#             if order.isbuy():
#                 self.log(
#                     'BUY EXECUTED, Size: %d, Price: %.2f, Cost: %.2f, Comm %.2f' %
#                     (order.executed.size,
#                      order.executed.price,
#                      order.executed.value,
#                      order.executed.comm))

#             else:  # Sell
#                 self.log('SELL EXECUTED, Size: %d, Price: %.2f, Cost: %.2f, Comm %.2f' %
#                          (order.executed.size,
#                           order.executed.price,
#                           order.executed.value,
#                           order.executed.comm))

#         elif order.status in [order.Canceled, order.Margin, order.Rejected]:
#             self.log('Order Canceled/Margin/Rejected')

        # Write down: no pending order
        self.order = None
    
    def notify_trade(self, trade):
        if not trade.isclosed:
            return

#         self.log('OPERATION PROFIT, GROSS %.2f, NET %.2f' %
#                  (trade.pnl, trade.pnlcomm))

    def next(self):
        #self.log('d0 {:f}'.format(self.data0.close[0]))
        # Check if an order is pending ... if yes, we cannot send a 2nd one
        if self.order:
            return
        
        # Check if we are in the market
        if not self.position:
            # Not yet ... we MIGHT BUY if ...
            if self.buy_signal[0]:
                # BUY, BUY, BUY!!! (with all possible default parameters)
                #self.log('BUY CREATE')
                # Keep track of the created order to avoid a 2nd order
                self.order = self.buy()
        else:
            # Already in the market ... we might sell
            if self.sell_signal[0]:
                # SELL, SELL, SELL!!! (with all possible default parameters)
                #self.log('SELL CREATE')

                # Keep track of the created order to avoid a 2nd order
                self.order = self.sell()
    def stop(self):
        self.log('(P {:2d}) Ending Value {:.2f}'.format(self.p.fastperiod, self.broker.getvalue()))

In [4]:
cerebro = bt.Cerebro()
cerebro.adddata(feed, name='etf300')
#cerebro.resampledata(feed, timeframe=bt.TimeFrame.Weeks)


cerebro.addstrategy(MACDStrategy)
#cerebro.optstrategy(MACDStrategy, fastperiod=range(5, 15))

# 起始资金
cerebro.broker.set_cash(100000.0)

# 手续费万5
cerebro.broker.setcommission(0.0005)

cerebro.broker.set_coc(True)
cerebro.broker.set_fundstartval(50)

#cerebro.writer = True

cerebro.addsizer(bt.sizers.AllInSizerInt, percents=99)

print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())

# cerebro.addanalyzer(bt.analyzers.AnnualReturn)
cerebro.addanalyzer(bt.analyzers.PyFolio)
# cerebro.addanalyzer(bt.analyzers.TimeDrawDown)
# cerebro.addanalyzer(bt.analyzers.TradeAnalyzer)
# cerebro.addanalyzer(bt.analyzers.SQN)
# cerebro.addanalyzer(bt.analyzers.VWR)

result = cerebro.run()

print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())

Starting Portfolio Value: 100000.00
2021-06-09, (P 10) Ending Value 584156.63
Final Portfolio Value: 584156.63


In [5]:
strat = result[0]
for a_name in strat.analyzers.getnames():
    strat.analyzers.getbyname(a_name).pprint()

OrderedDict([('returns',
              OrderedDict([(datetime.datetime(2005, 4, 8, 0, 0), 0.0),
                           (datetime.datetime(2005, 4, 11, 0, 0), 0.0),
                           (datetime.datetime(2005, 4, 12, 0, 0), 0.0),
                           (datetime.datetime(2005, 4, 13, 0, 0), 0.0),
                           (datetime.datetime(2005, 4, 14, 0, 0), 0.0),
                           (datetime.datetime(2005, 4, 15, 0, 0), 0.0),
                           (datetime.datetime(2005, 4, 18, 0, 0), 0.0),
                           (datetime.datetime(2005, 4, 19, 0, 0), 0.0),
                           (datetime.datetime(2005, 4, 20, 0, 0), 0.0),
                           (datetime.datetime(2005, 4, 21, 0, 0), 0.0),
                           (datetime.datetime(2005, 4, 22, 0, 0), 0.0),
                           (datetime.datetime(2005, 4, 25, 0, 0), 0.0),
                           (datetime.datetime(2005, 4, 26, 0, 0), 0.0),
                           (datetime.dat

In [6]:
strat = result[0]
pyfoliozer = strat.analyzers.getbyname('pyfolio')
returns, positions, transactions, gross_lev = pyfoliozer.get_pf_items()

import pyfolio as pf

import warnings
warnings.filterwarnings("ignore")

pf.create_full_tear_sheet(
    returns,
    positions=positions,
    transactions=transactions,
#     live_start_date='2018-01-01',  # This date is sample specific
    round_trips=True)

ModuleNotFoundError: No module named 'pyfolio'

In [ ]:
cerebro.plot(iplot=False,style='candle')